# Preparación del ambiente

Se comprueba la lectura de los dos CSV en Raw y se crean los esquemas del proyecto en Unity Catalog. El parámetro `raw_path` apunta a la carpeta `raw/bitcoin` del contenedor `delta-lake`, accesible con identidad administrada.

In [ ]:
dbutils.widgets.text("catalog", "catalog_smartdata", "Catalog")
dbutils.widgets.text("raw_path", "", "Raw path")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
raw_path = dbutils.widgets.get("raw_path").strip().rstrip("/")

## Lectura de los archivos Raw

Cada consulta muestra cinco filas del CSV de origen.

In [ ]:
binance_path = f"{raw_path}/binance/btcusdt_1m_2022_2025.csv"
prices = spark.read.option("header", True).csv(binance_path)
display(prices.select("open_time", "close", "volume").limit(5))

In [ ]:
sentiment_path = f"{raw_path}/alternative-me/fear_greed_2022_2025.csv"
sentiment = spark.read.option("header", True).csv(sentiment_path)
display(sentiment.select("timestamp", "value", "value_classification").limit(5))

## Esquemas del proyecto

Los esquemas se crean dentro del catálogo indicado. Las capas Bronze, Silver y Gold se cargan en sus respectivos notebooks.

In [ ]:
spark.sql("USE CATALOG IDENTIFIER(:catalog)", args={"catalog": catalog})
spark.sql("CREATE SCHEMA IF NOT EXISTS bitcoin_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS bitcoin_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS bitcoin_gold")

display(spark.sql("SHOW SCHEMAS LIKE 'bitcoin_*'"))